In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\chivi\OneDrive\Documentos\GitHub\TFG_software


In [2]:
# Load municipal household disposable income
rfdb_raw = pd.read_csv(
    RAW_DATA / "rfdb_municipalities.csv",
    sep=";",
    encoding="utf-8-sig"
)

# Convert numeric values, including decimal commas in the index indicator
rfdb_raw["valor"] = pd.to_numeric(
    rfdb_raw["valor"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

print("RFDB dataset shape:", rfdb_raw.shape)
print("Available years:", sorted(rfdb_raw["año"].unique()))
display(rfdb_raw.head())

RFDB dataset shape: (19782, 6)
Available years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,año,municipio,concepto,indicador,estado,valor
0,2010,Abrera,renta disponible bruta de los hogares,valor (miles €),NaN,197787.0
1,2010,Abrera,renta disponible bruta de los hogares,por habitante (€),NaN,17241.0
2,2010,Abrera,renta disponible bruta de los hogares,por habitante (índice Cataluña=100),NaN,101.8
3,2010,Agramunt,renta disponible bruta de los hogares,valor (miles €),NaN,78123.0
4,2010,Agramunt,renta disponible bruta de los hogares,por habitante (€),NaN,13956.0


In [3]:
rfdb_2023 = rfdb_raw.loc[
    (rfdb_raw["año"] == 2023)
    & (rfdb_raw["indicador"] == "por habitante (€)")
    & (rfdb_raw["concepto"] == "renta disponible bruta de los hogares"),
    ["municipio", "valor", "estado"]
].copy()

rfdb_2023 = rfdb_2023.rename(
    columns={"valor": "rfdb_per_capita_eur"}
)

assert not rfdb_2023["municipio"].duplicated().any()

print("Municipalities with RFDB data:", len(rfdb_2023))
display(
    rfdb_2023.sort_values(
        "rfdb_per_capita_eur", ascending=False
    ).head(10)
)

Municipalities with RFDB data: 471


,municipio,rfdb_per_capita_eur,estado
18406,Alella,35425.0,p
19003,Matadepera,34766.0,p
19408,Sant Just Desvern,33256.0,p
18622,Cabrils,31280.0,p
19336,Sant Cugat del Vallès,30434.0,p
19597,Tiana,28136.0,p
19588,Teià,28108.0,p
18619,Cabrera de Mar,27805.0,p
18859,Fornells de la Selva,27805.0,p
18538,Begues,27588.0,p


In [4]:
seasonal_raw = pd.read_csv(
    RAW_DATA / "seasonal_population_quarterly.csv",
    skiprows=7,
    encoding="utf-8-sig",
    dtype={"Codi": str},
    na_values=["..", "...", ""]
)

print("Seasonal dataset shape:", seasonal_raw.shape)
print("Columns:")
print(seasonal_raw.columns.tolist())

display(seasonal_raw.head())

Seasonal dataset shape: (485, 9)
Columns:
['Codi', 'Nom', 'Població estacional ETCA. T1', 'Població estacional ETCA. T2', 'Població estacional ETCA. T3', 'Població estacional ETCA. T4', 'Població estacional ETCA. Total', 'Població resident', 'Població ETCA']


,Codi,Nom,Població estacional ETCA. T1,Població estacional ETCA. T2,Població estacional ETCA. T3,Població estacional ETCA. T4,Població estacional ETCA. Total,Població resident,Població ETCA
0,080018,Abrera,83.0,-40.0,-603.0,39.0,-130.0,12981,12851.0
1,250030,Agramunt,-281.0,-369.0,-554.0,-264.0,-366.0,5592,5226.0
2,080142,Aiguafreda,-192.0,-235.0,-283.0,-200.0,-227.0,2582,2355.0
3,430017,Aiguamúrcia,4.0,32.0,129.0,31.0,49.0,965,1014.0
4,170025,Aiguaviva,279.0,291.0,169.0,293.0,258.0,800,1058.0


In [5]:
seasonal = seasonal_raw.rename(columns={
    "Nom": "municipio",
    "Població estacional ETCA. T1": "seasonal_t1",
    "Població estacional ETCA. T2": "seasonal_t2",
    "Població estacional ETCA. T3": "seasonal_t3",
    "Població estacional ETCA. T4": "seasonal_t4",
    "Població estacional ETCA. Total": "seasonal_total",
    "Població resident": "resident_population_2024",
    "Població ETCA": "population_etca_2024"
})

seasonal_columns = [
    "seasonal_t1",
    "seasonal_t2",
    "seasonal_t3",
    "seasonal_t4",
    "seasonal_total",
    "resident_population_2024",
    "population_etca_2024"
]

for column in seasonal_columns:
    seasonal[column] = pd.to_numeric(
        seasonal[column], errors="coerce"
    )

seasonal = seasonal[
    ["municipio"] + seasonal_columns
].copy()

# Harmonise known municipal naming differences
seasonal["municipio"] = seasonal["municipio"].replace({
    "Palamós i Sant Joan": "Palamós"
})

assert not seasonal["municipio"].duplicated().any()

display(seasonal.head())

,municipio,seasonal_t1,seasonal_t2,seasonal_t3,seasonal_t4,seasonal_total,resident_population_2024,population_etca_2024
0,Abrera,83.0,-40.0,-603.0,39.0,-130.0,12981,12851.0
1,Agramunt,-281.0,-369.0,-554.0,-264.0,-366.0,5592,5226.0
2,Aiguafreda,-192.0,-235.0,-283.0,-200.0,-227.0,2582,2355.0
3,Aiguamúrcia,4.0,32.0,129.0,31.0,49.0,965,1014.0
4,Aiguaviva,279.0,291.0,169.0,293.0,258.0,800,1058.0


In [6]:
demographic = pd.read_csv(
    PROCESSED_DATA / "demographic_municipality_stats_2025.csv",
    encoding="utf-8-sig"
)

# Merge the three data sources
analysis_df = (
    demographic
    .merge(
        rfdb_2023[["municipio", "rfdb_per_capita_eur"]],
        on="municipio",
        how="left",
        validate="one_to_one"
    )
    .merge(
        seasonal,
        on="municipio",
        how="left",
        validate="one_to_one"
    )
)

print("Integrated dataset shape:", analysis_df.shape)
print("Municipalities:", analysis_df["municipio"].nunique())

display(analysis_df.head())

Integrated dataset shape: (948, 12)
Municipalities: 948


,municipio,population_total,population_60_plus,pct_60_plus,rfdb_per_capita_eur,seasonal_t1,seasonal_t2,seasonal_t3,seasonal_t4,seasonal_total,resident_population_2024,population_etca_2024
0,Catalunya,8124126,2106500,25.928943,20789.0,NaN,NaN,NaN,NaN,NaN,8012231.0,NaN
1,Barcelona,1713247,460936,26.904235,25898.0,135469.0,138045.0,-28400.0,144003.0,96891.0,1686208.0,1783099.0
2,"Hospitalet de Llobregat, l'",292161,70588,24.160651,17591.0,-14554.0,-18757.0,-31477.0,-15650.0,-20110.0,282299.0,262189.0
3,Badalona,230642,59881,25.962747,19114.0,-14479.0,-18044.0,-23461.0,-15059.0,-17760.0,226219.0,208459.0
4,Sabadell,224589,57424,25.568483,20303.0,-11620.0,-15061.0,-22768.0,-11933.0,-15388.0,221564.0,206176.0


In [7]:
coverage = pd.DataFrame({
    "indicator": [
        "RFDB per capita",
        "Seasonal population"
    ],
    "municipalities_available": [
        analysis_df["rfdb_per_capita_eur"].notna().sum(),
        analysis_df["seasonal_total"].notna().sum()
    ]
})

coverage["coverage_pct"] = (
    100 * coverage["municipalities_available"]
    / len(analysis_df)
)

display(coverage.round(2))

,indicator,municipalities_available,coverage_pct
0,RFDB per capita,471,49.68
1,Seasonal population,484,51.05


In [ ]:
costa_brava_municipalities = [
    "Blanes",
    "Lloret de Mar",
    "Tossa de Mar",
    "Santa Cristina d'Aro",
    "Sant Feliu de Guíxols",
    "Castell d'Aro, Platja d'Aro i s'Agaró",
    "Calonge i Sant Antoni",
    "Palamós",
    "Palafrugell",
    "Mont-ras",
    "Begur",
    "Pals",
    "Torroella de Montgrí",
    "Escala, l'",
    "Sant Pere Pescador",
    "Castelló d'Empúries",
    "Roses",
    "Cadaqués",
    "Port de la Selva, el",
    "Llançà",
    "Colera",
    "Portbou",
]
costa_brava_stats = analysis_df.loc[
    analysis_df["municipio"].isin(costa_brava_municipalities)
].copy()

costa_brava_stats = costa_brava_stats.sort_values(
    "population_60_plus",
    ascending=False
)

display(
    costa_brava_stats[
        [
            "municipio",
            "population_total",
            "population_60_plus",
            "pct_60_plus",
            "rfdb_per_capita_eur",
            "seasonal_total",
            "resident_population_2024",
            "population_etca_2024"
        ]
    ].round(2)
)